In [1]:

import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import RF_PARAM_5G, NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt("data/random_seeds.csv", dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ["toa_pps", "toa_cir", "toa_cov", "campaign_id"]
df["measurements_matrix"] = df["measurements_matrix"].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

operator_choice = None

selected_campaigns = None
rf_param = RF_PARAM_5G.RSRQ

# Data filtering
df = filter_dataframe(
    df=df,
    operators=operator_choice,
    include_columns=[
        "pci",
        "beam_index",
        "nr_arfcn",
        "operator_id",
        "sinr",
        "rsrq"
    ],
    campaigns=selected_campaigns,
)




Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


In [2]:
df['campaign_id'].value_counts()



campaign_id
72    530
8     529
75    528
66    520
9     505
     ... 
70    341
40    336
47    332
34    330
17    290
Name: count, Length: 79, dtype: int64

In [3]:
df.shape

(33537, 4)

In [4]:
def count_len(mat):
    return mat.shape[0]


df['size'] = df['measurements_matrix'].apply(count_len)

df['size'].mean()

np.float64(157.46435280436532)

In [5]:
df['size'].sum()

np.int64(5280882)

In [6]:
df['campaign_id'].unique().shape

(79,)

In [13]:
import folium
import pandas as pd


def geo_plot_points(df: pd.DataFrame):
    """
    Plots given locations to a map (OpenStreetMap) that is viewable in broswer.
    Generates a file called 'map.html' in the current working directory.
    :param df:
    """
    # Create a map centered around the mean location
    m = folium.Map(location=[df["lat"].mean(), df["lng"].mean()], zoom_start=12)

    # Add CircleMarkers to the map
    for _, row in df.iterrows():
        folium.CircleMarker(
            location=[row["lat"], row["lng"]],
            radius=1,  # Size of the marker
            color="blue",  # Border color of the marker
            fill=True,
            fill_color="blue",  # Fill color of the marker
            fill_opacity=0.6,
        ).add_to(m)

    # Save the map as an HTML file and open it in the browser
    m.save("map.html")


geo_plot_points(df.sample(3000))
